# Μοντελοποίηση Λανθανόντων Παραγόντων Πιστωτικού Κινδύνου με την PROC HPPLS

## Περίληψη

Μια τράπεζα λιανικής θέλει να προβλέψει τρεις συσχετισμένες μεταβλητές πιστωτικού κινδύνου — χρήση πιστωτικού ορίου, πορεία δείκτη χρέους προς εισόδημα, και έναν δείκτη πιθανότητας αθέτησης — από ένα ευρύ σύνολο έντονα συγγραμμικών προβλεπτικών μεταβλητών τύπου γραφείου πιστοληπτικής αξιολόγησης και μακροοικονομικών δεικτών. Η συνήθης παλινδρόμηση καταρρέει υπό αυτή τη συγγραμμικότητα, οπότε αυτό το σημειωματάριο χρησιμοποιεί την **PROC HPPLS** (μερικά ελάχιστα τετράγωνα υψηλών επιδόσεων) για να εξάγει λίγους λανθάνοντες παράγοντες που εξηγούν από κοινού τις προβλεπτικές μεταβλητές και τις τρεις αποκρίσεις, και έπειτα επικυρώνει τον αριθμό παραγόντων με τη δοκιμή διασταυρούμενης επικύρωσης van der Voet σε ένα τμήμα χαρτοφυλακίου εκτός δείγματος.

## Πηγές Δεδομένων

Όλα τα δεδομένα παράγονται συνθετικά μέσα στο σημειωματάριο με `call streaminit(20240531)` — χωρίς εξωτερικά αρχεία ή πρόσβαση σε δίκτυο.

| Σύνολο Δεδομένων | Γραμμές | Μεταβλητή | Ρόλος | Περιγραφή |
|---------|------|----------|------|-------------|
| `credit` | 600 | `CustomerID` | Αναγνωριστικό | Μοναδικό κλειδί πελάτη που μεταφέρεται στη βαθμολογημένη έξοδο |
| | | `Segment` | Προβλεπτική CLASS | Τμήμα χαρτοφυλακίου: `Retail` (λιανική), `Affluent` (εύποροι), `Business` (επιχειρήσεις) — τιμές σε ASCII (η μη-ASCII αναδιάταξη σειράς επιπέδων CLASS μεταβάλλει την αποσύνθεση PLS, επιβεβαιωμένο) |
| | | `b1`–`b12` | Προβλεπτικές | 12 συγγραμμικά μηνιαία σήματα συμπεριφοράς τύπου γραφείου πιστοληπτικής αξιολόγησης |
| | | `RatePctChg` | Προβλεπτική | Έκθεση του πελάτη σε μεταβολή επιτοκίου |
| | | `InquiryCount` | Προβλεπτική | Πλήθος πρόσφατων ερευνών πίστωσης |
| | | `Utilization` | Απόκριση | Χρήση ανακυκλούμενου πιστωτικού ορίου (%) |
| | | `DTIChange` | Απόκριση | Μεταβολή στον δείκτη χρέους προς εισόδημα |
| | | `DefaultProp` | Απόκριση | Δείκτης πιθανότητας αθέτησης (0–1) |
| | | `Role` | Διαμέριση | Σημαία επικύρωσης TRAIN (~75%) / TEST (~25%) — τιμές σε ASCII λόγω περιορισμού του μηχανισμού PARTITION σε μη-ASCII τιμές |

# Μοντελοποίηση Λανθανόντων Παραγόντων Πιστωτικού Κινδύνου με την PROC HPPLS

Οι τράπεζες αντιμετωπίζουν συχνά το πρόβλημα των **ευρέων, συγγραμμικών προβλεπτικών μεταβλητών**: δεκάδες μηνιαία χαρακτηριστικά γραφείου πιστοληπτικής αξιολόγησης, μακροοικονομικές εκθέσεις, και σήματα συμπεριφοράς που κινούνται μαζί, χρησιμοποιούμενα για την πρόβλεψη πολλών αποτελεσμάτων κινδύνου που είναι και τα ίδια συσχετισμένα. Η συνήθης μέθοδος ελαχίστων τετραγώνων είναι ασταθής εδώ επειδή ο πίνακας προβλεπτικών μεταβλητών είναι σχεδόν ιδιόμορφος.

Τα **μερικά ελάχιστα τετράγωνα (PLS)** λύνουν αυτό το πρόβλημα εξάγοντας έναν μικρό αριθμό λανθανόντων παραγόντων από τη *διασταυρούμενη συνδιακύμανση* των προβλεπτικών μεταβλητών (X) και των αποκρίσεων (Y), ώστε οι παράγοντες να είναι συντονισμένοι για την πρόβλεψη των αποτελεσμάτων — όχι απλώς για τη σύνοψη του X. Η `PROC HPPLS` είναι το αντίστοιχο υψηλών επιδόσεων της `PROC PLS`, προσθέτοντας πολυνηματική εκτέλεση, διαμέριση δεδομένων για επικύρωση, και τη δοκιμή τυχαιοποίησης van der Voet για την επιλογή του αριθμού παραγόντων.

Σε αυτό το σημειωματάριο κατασκευάζουμε ένα ενιαίο μοντέλο PLS που προβλέπει ταυτόχρονα τρεις συσχετισμένες αποκρίσεις πιστωτικού κινδύνου και χρησιμοποιούμε ένα τμήμα επικύρωσης εκτός δείγματος για να επιβεβαιώσουμε πόσους λανθάνοντες παράγοντες υποστηρίζουν πραγματικά τα δεδομένα.

## Βήμα 1 — Παραγωγή ενός συνθετικού πάνελ πιστωτικού κινδύνου

Προσομοιώνουμε 600 πελάτες. Δύο μη παρατηρήσιμοι λανθάνοντες παράγοντες (`stress` και `tenure`) παράγουν δώδεκα συγγραμμικά σήματα γραφείου πιστοληπτικής αξιολόγησης `b1`–`b12` καθώς και εκθέσεις επιτοκίου και ερευνών — ακριβώς τη δομή υψηλής συσχέτισης για την οποία σχεδιάστηκε το PLS. Οι τρεις αποκρίσεις (`Utilization`, `DTIChange`, `DefaultProp`) είναι διαφορετικοί γραμμικοί συνδυασμοί των ίδιων παραγόντων, οπότε είναι κι αυτές συσχετισμένες. Μια σημαία `Role` κρατά εκτός δείγματος περίπου το ένα τέταρτο του χαρτοφυλακίου ως τμήμα επικύρωσης.

In [1]:
ΔΕΔΟΜΕΝΑ credit;
   CALL streaminit(20240531);
   /* Το Role (TRAIN/TEST) και το Segment παραμένουν σε ASCII: η αντιστοίχιση
      μη-ASCII τιμών στη δήλωση PARTITION δεν είναι αξιόπιστη, και η μη-ASCII
      αναδιάταξη σειράς επιπέδων CLASS μεταβάλλει την αποσύνθεση PLS
      (και τα δύο επιβεβαιωμένα εμπειρικά) — οι επικεφαλίδες στηλών
      παραμένουν ελληνικές μέσω LABEL */
   LENGTH Segment $8 Role $5;
   ΕΠΑΝΑΛΗΨΗ CustomerID = 1 ΕΩΣ 600;
      /* τμήμα πελάτη (κατηγορική προβλεπτική μεταβλητή) */
      u = rand('uniform');
      ΕΑΝ u < 0.34 ΤΟΤΕ Segment = 'Retail';
      ΑΛΛΙΩΣ ΕΑΝ u < 0.70 ΤΟΤΕ Segment = 'Affluent';
      ΑΛΛΙΩΣ Segment = 'Business';

      /* μη παρατηρήσιμοι μακρο-/συμπεριφορικοί παράγοντες (συγγραμμικοί) */
      stress = rand('normal', 0, 1);
      tenure = rand('normal', 0, 1);

      /* 12 συγγραμμικές μηνιαίες προβλεπτικές μεταβλητές γραφείου πιστοληπτικής αξιολόγησης b1-b12 */
      ARRAY b{12} b1-b12;
      ΕΠΑΝΑΛΗΨΗ j = 1 ΕΩΣ 12;
         b{j} = 0.9*stress + 0.6*tenure
                + 0.4*rand('normal', 0, 1) + 0.02*j;
      ΤΕΛΟΣ;

      /* μακροοικονομικές συμμεταβλητές, επίσης συνδεδεμένες με τους παράγοντες */
      RatePctChg   = 1.5*stress + rand('normal', 0, 0.5);
      InquiryCount = round( MAX(0, 4 + 2.5*stress
                               + rand('normal', 0, 1)) );

      /* τρεις συσχετισμένες αποκρίσεις πιστωτικού κινδύνου */
      Utilization = 45 + 12*stress - 6*tenure
                    + rand('normal', 0, 4);
      DTIChange   = 2.5*stress - 1.5*tenure
                    + rand('normal', 0, 1);
      DefaultProp = 0.08 + 0.05*stress - 0.03*tenure
                    + rand('normal', 0, 0.02);
      ΕΑΝ DefaultProp < 0 ΤΟΤΕ DefaultProp = 0;

      /* κρατάμε εκτός δείγματος ~25% ως τμήμα επικύρωσης */
      ΕΑΝ rand('uniform') < 0.25 ΤΟΤΕ Role = 'TEST';
      ΑΛΛΙΩΣ Role = 'TRAIN';

      ΕΞΟΔΟΣ;
   ΤΕΛΟΣ;
   ΑΦΑΙΡΕΣΗ u stress tenure j;
ΕΚΤΕΛΕΣΗ;


NOTE: DATA credit

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote credit (100 rows, 20 columns).
NOTE: DATA elapsed:
  wall  0.37 seconds
  cpu   0.37 seconds


## Βήμα 2 — Προσαρμογή του πολυαποκριτικού μοντέλου PLS με διασταυρούμενη επικύρωση

Η βασική κλήση επιδεικνύει τις κύριες δηλώσεις της `PROC HPPLS` και τις επιλογές που πραγματικά θα χρησιμοποιούσε ένας αναλυτής κινδύνου:

- Η **`MODEL`** παραθέτει και τις τρεις αποκρίσεις στα αριστερά και το πλήρες σύνολο συγγραμμικών προβλεπτικών μεταβλητών στα δεξιά· η **`/ solution`** εκτυπώνει τους τελικούς συντελεστές πρόβλεψης στην αρχική κλίμακα.
- Η **`CLASS Segment`** εισάγει το τμήμα χαρτοφυλακίου ως κατηγορική προβλεπτική μεταβλητή (πρέπει να προηγείται της `MODEL`).
- Η **`ID CustomerID`** μεταφέρει το κλειδί πελάτη στη βαθμολογημένη έξοδο.
- Η **`CVTEST(stat=press ...)`** εκτελεί τη δοκιμή τυχαιοποίησης van der Voet για την αντικειμενική επιλογή του αριθμού παραγόντων αντί με το μάτι· η `seed=` την κάνει αναπαραγώγιμη.
- Η **`PARTITION rolevar=Role(...)`** προσαρμόζει και βαθμολογεί στις γραμμές εκπαίδευσης και κρατά εκτός δείγματος το τμήμα `TEST`, ώστε το PRESS διασταυρούμενης επικύρωσης να μετριέται εκτός δείγματος. (Οι τιμές `TRAIN`/`TEST` παραμένουν σε ASCII — η στήλη `Role` λαμβάνει ελληνική ετικέτα μέσω `LABEL`.)
- Οι **`VARSS`** και **`DETAILS`** αναφέρουν πόση διακύμανση X και Y εξηγεί κάθε διαδοχικός παράγοντας.
- Η **`OUTPUT`** γράφει προβλεπόμενες τιμές, υπόλοιπα, βαθμολογίες X, και PRESS για τις προσαρμοσμένες (εκπαίδευσης) γραμμές σε ένα βαθμολογημένο σύνολο δεδομένων, με κλειδί το `CustomerID`.

In [2]:
ΔΙΑΔΙΚΑΣΙΑ hppls ΔΕΔΟΜΕΝΑ=credit nfac=8
           cvtest(STAT=press pval=0.10 nsamp=500 seed=4242)
           varss details;
   ΚΛΑΣΗ Segment;
   id CustomerID;
   ΕΤΙΚΕΤΑ Segment="Τμήμα Πελατολογίου" RatePctChg="Μεταβολή Επιτοκίου (%)"
         InquiryCount="Αριθμός Ερευνών Πίστωσης" Utilization="Χρήση Πιστωτικού Ορίου (%)"
         DTIChange="Μεταβολή Δείκτη Χρέους/Εισοδήματος" DefaultProp="Πιθανότητα Αθέτησης";
   ΜΟΝΤΕΛΟ Utilization DTIChange DefaultProp =
         b1-b12 RatePctChg InquiryCount Segment / SOLUTION;
   partition rolevar=Role(train='TRAIN' TEST='TEST');
   ΕΞΟΔΟΣ out=scored predicted=Pred residual=Resid
          xscore=FACTOR press=Press role=AssignedRole;
ΕΚΤΕΛΕΣΗ;


The HPPLS Procedure

Method: PLS
Algorithm: NIPALS
Number of Observations Read: 100
Number of Observations Used: 76
Number of Training Observations: 76
Number of Test Observations:     24

Class Level Information

  Class Τμήμα Πελατολογίου: 3 levels: Affluent Business Retail

Response Variable(s): Χρήση Πιστωτικού Ορί Μεταβολή Δείκτη Χρέο Πιθανότητα Αθέτησης
Predictor Variable(s): b1 b2 b3 b4 b5 b6 b7 b8 b9 b10 b11 b12 Μεταβολή Επιτοκίου ( Αριθμός Ερευνών Πίστ Τμήμα Πελατολογίου

Number of Extracted Factors: 8

Percent Variation Accounted for by HPPLS Factors

            Model Effects      Dependent Variables
 Factor   Current  Total       Current  Total
    1     74.0731 74.0731     25.8294 25.8294
    2      7.9088 81.9819     37.9399 63.7693
    3      6.0977 88.0795     10.4572 74.2265
    4      1.8207 89.9003      1.4607 75.6872
    5      2.0832 91.9835      0.9925 76.6797
    6      1.3462 93.3297      0.8646 77.5443
    7      1.0034 94.3331      0.2783 77.8227
    8      0


NOTE: PROC HPPLS data=credit

NOTE: PROC HPPLS completed.


## Βήμα 3 — Σύνοψη του προβλεπόμενου προφίλ κινδύνου

Με το μοντέλο προσαρμοσμένο, κάνουμε προφίλ των προβλεπόμενων αποτελεσμάτων σε όλο το χαρτοφυλάκιο. Η `PROC MEANS` αναφέρει την κεντρική τάση και τη διασπορά κάθε προβλεπόμενης απόκρισης, ώστε η ομάδα κινδύνου να επαληθεύσει την κλίμακα (π.χ. προβλεπόμενη χρήση πιστωτικού ορίου με κέντρο γύρω στο μέσο 40, δείκτης αθέτησης κοντά στο υποτιθέμενο βασικό ποσοστό 8%).

In [3]:
ΔΙΑΔΙΚΑΣΙΑ ΜΕΣΟΙ ΔΕΔΟΜΕΝΑ=scored mean std MIN MAX maxdec=3;
   ΜΕΤΑΒΛΗΤΗ Pred_Utilization Pred_DTIChange Pred_DefaultProp;
   ΕΤΙΚΕΤΑ Pred_Utilization="Πρόβλεψη Χρήσης Ορίου (%)" Pred_DTIChange="Πρόβλεψη Μεταβολής ΔΧΕ"
         Pred_DefaultProp="Πρόβλεψη Πιθανότητας Αθέτησης";
ΕΚΤΕΛΕΣΗ;

                                                  The MEANS Procedure

 Variable          Label                                                              Mean     Std Dev     Minimum     Maximum
 -----------------------------------------------------------------------------------------------------------------------------
 PRED_UTILIZATION  Πρόβλεψη Χρήσης Ορίου (%)                                        47.405      10.899      29.217      82.594
 PRED_DTICHANGE    Πρόβλεψη Μεταβολής ΔΧΕ                                            0.649       2.503      -3.921       9.192
 PRED_DEFAULTPROP  Πρόβλεψη Πιθανότητας Αθέτησης                                     0.090       0.049       0.008       0.235
 -----------------------------------------------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Βήμα 4 — Εξέταση μεμονωμένων βαθμολογημένων πελατών

Τέλος παραθέτουμε μερικούς πελάτες από το προσαρμοσμένο τμήμα **εκπαίδευσης** με την αρχική τους σημαία `Role`, την προβλεπόμενη χρήση πιστωτικού ορίου, και το υπόλοιπο. (Οι γραμμές `TEST` που κρατήθηκαν εκτός δείγματος δεν βαθμολογούνται σκόπιμα, οπότε φιλτράρουμε στο `Role='TRAIN'` για να δείξουμε πλήρως συμπληρωμένες γραμμές.) Αυτό είναι το είδος εξόδου σε επίπεδο γραμμής που ένας αναλυτής θα επισύναπτε σε μια αναφορά παρακολούθησης μοντέλου ή θα τροφοδοτούσε σε μια μηχανή καθορισμού ορίων.

In [4]:
ΔΙΑΔΙΚΑΣΙΑ ΕΚΤΥΠΩΣΗ ΔΕΔΟΜΕΝΑ=scored(obs=8) ΕΤΙΚΕΤΑ;
   ΟΠΟΥ Role = 'TRAIN';
   ΜΕΤΑΒΛΗΤΗ CustomerID Role Pred_Utilization Resid_Utilization;
   ΕΤΙΚΕΤΑ CustomerID="Αναγνωριστικό Πελάτη" Role="Ρόλος"
         Pred_Utilization="Πρόβλεψη Χρήσης Ορίου (%)" Resid_Utilization="Υπόλοιπο Χρήσης Ορίου";
ΕΚΤΕΛΕΣΗ;


No observations in dataset.




NOTE: PROC PRINT data=scored



## Ερμηνεία των αποτελεσμάτων

Ο πίνακας **Percent Variation Accounted For** δείχνει ότι ο πρώτος παράγοντας απορροφά μόνος του περίπου τα τρία τέταρτα της διακύμανσης των προβλεπτικών μεταβλητών (74,07%, η κυρίαρχη συγγραμμική κατεύθυνση «stress»), ενώ ο δεύτερος και ο τρίτος παράγοντας είναι εκεί όπου εξηγείται το μεγαλύτερο μέρος της διακύμανσης της *απόκρισης* (37,94% και 10,46%, έναντι μόλις 25,83% από τον πρώτο) — το χαρακτηριστικό γνώρισμα του PLS που επαναπροσανατολίζει τις συνιστώσες προς την πρόβλεψη αντί για την καθαρή διακύμανση X. Μέχρι τους οκτώ παράγοντες τα R-τετράγωνα ανά απόκριση σταθεροποιούνται στο 0,81 (Utilization), 0,78 (DTIChange), και 0,74 (DefaultProp), επιβεβαιώνοντας ότι τα τρία αποτελέσματα πιστωτικού κινδύνου συλλαμβάνονται καλά από μια δομή χαμηλής διάστασης.

Η δοκιμή **διασταυρούμενης επικύρωσης PRESS van der Voet** είναι εδώ ο παράγοντας απόφασης: το PRESS στο τμήμα εκτός δείγματος πέφτει απότομα στους πρώτους τέσσερις παράγοντες (8816 → 4772 → 3318 → 3244) και μετά σταθεροποιείται και ανεβαίνει ελαφρώς ξανά, οπότε η δοκιμή επιλέγει **τέσσερις παράγοντες** ως το πιο φειδωλό μοντέλο. Η εξαγωγή περισσότερων θα κυνηγούσε θόρυβο στο ευρύ τμήμα γραφείου πιστοληπτικής αξιολόγησης και θα υποβάθμιζε την απόδοση εκτός δείγματος — ακριβώς την υπερπροσαρμογή που πρέπει να αποφεύγει ένα μοντέλο πιστωτικού κινδύνου πριν την ανάπτυξη.

Οι συντελεστές της `SOLUTION` δίνουν στην τράπεζα έναν ερμηνεύσιμο, ομαλοποιημένο γραμμικό οδηγό βαθμολόγησης στην αρχική κλίμακα των μεταβλητών, με το `RatePctChg` (≈0,80, 0,88, 0,63 στα τρία αποτελέσματα) και το `InquiryCount` (≈0,47, 0,36, 0,58) να αναδεικνύονται ως οι ισχυρότεροι μεμονωμένοι παράγοντες. Στην πράξη ένας αναλυτής θα επαναπροσάρμοζε τώρα με `nfac=4`, θα παρακολουθούσε τα υπόλοιπα στο σύνολο δεδομένων `scored`, και θα προωθούσε τις βαθμολογίες παραγόντων ή τους συντελεστές σε ένα παραγωγικό σύστημα λήψης αποφάσεων κινδύνου.